#Phase 1: Dataset Overview & Assessment


In [47]:
import pandas as pd

In [48]:
df1 = pd.read_csv(r"/content/QVI_purchase_behaviour.csv")

In [49]:
df2 = pd.read_excel(r"/content/QVI_transaction_data.xlsx")

In [50]:
df_purchase = df1.copy()
df_transaction = df2.copy()

In [51]:
df_transaction.describe()

,DATE,STORE_NBR,LYLTY_CARD_NBR,TXN_ID,PROD_NBR,PROD_QTY,TOT_SALES
count,264836.000000,264836.00000,2.648360e+05,2.648360e+05,264836.000000,264836.000000,264836.000000
mean,43464.036260,135.08011,1.355495e+05,1.351583e+05,56.583157,1.907309,7.304200
std,105.389282,76.78418,8.057998e+04,7.813303e+04,32.826638,0.643654,3.083226
min,43282.000000,1.00000,1.000000e+03,1.000000e+00,1.000000,1.000000,1.500000
25%,43373.000000,70.00000,7.002100e+04,6.760150e+04,28.000000,2.000000,5.400000
50%,43464.000000,130.00000,1.303575e+05,1.351375e+05,56.000000,2.000000,7.400000
75%,43555.000000,203.00000,2.030942e+05,2.027012e+05,85.000000,2.000000,9.200000
max,43646.000000,272.00000,2.373711e+06,2.415841e+06,114.000000,200.000000,650.000000


In [52]:
df_transaction.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 264836 entries, 0 to 264835
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   DATE            264836 non-null  int64  
 1   STORE_NBR       264836 non-null  int64  
 2   LYLTY_CARD_NBR  264836 non-null  int64  
 3   TXN_ID          264836 non-null  int64  
 4   PROD_NBR        264836 non-null  int64  
 5   PROD_NAME       264836 non-null  object 
 6   PROD_QTY        264836 non-null  int64  
 7   TOT_SALES       264836 non-null  float64
dtypes: float64(1), int64(6), object(1)
memory usage: 16.2+ MB


In [53]:
df_transaction.head()

,DATE,STORE_NBR,LYLTY_CARD_NBR,TXN_ID,PROD_NBR,PROD_NAME,PROD_QTY,TOT_SALES
0,43390,1,1000,1,5,Natural Chip Compny SeaSalt175g,2,6.0
1,43599,1,1307,348,66,CCs Nacho Cheese 175g,3,6.3
2,43605,1,1343,383,61,Smiths Crinkle Cut Chips Chicken 170g,2,2.9
3,43329,2,2373,974,69,Smiths Chip Thinly S/Cream&Onion 175g,5,15.0
4,43330,2,2426,1038,108,Kettle Tortilla ChpsHny&Jlpno Chili 150g,3,13.8


#Phase 2: Data Cleaning  

In [54]:
df_purchase.duplicated().sum()

np.int64(0)

In [55]:
df_transaction.duplicated().sum()
df_transaction[df_transaction.duplicated(keep=False)]

,DATE,STORE_NBR,LYLTY_CARD_NBR,TXN_ID,PROD_NBR,PROD_NAME,PROD_QTY,TOT_SALES
124843,43374,107,107024,108462,45,Smiths Thinly Cut Roast Chicken 175g,2,6.0
124845,43374,107,107024,108462,45,Smiths Thinly Cut Roast Chicken 175g,2,6.0


In [56]:
df_transaction.drop_duplicates(inplace=True)

In [57]:
df_transaction["DATE"].dtype

dtype('int64')

In [58]:
# Convert Excel serial dates to datetime
df_transaction["DATE"] = pd.to_datetime(
    df_transaction["DATE"],
    unit="D",
    origin="1899-12-30"
)

In [59]:
df_transaction["DATE"].dtype

dtype('<M8[ns]')

In [60]:
# Check the first few dates
df_transaction["DATE"].head()

,DATE
0,2018-10-17
1,2019-05-14
2,2019-05-20
3,2018-08-17
4,2018-08-18


In [61]:
# Check the date range
df_transaction["DATE"].agg(["min", "max"])

,DATE
min,2018-07-01
max,2019-06-30


In [62]:
# Check unique years
df_transaction["DATE"].dt.year.unique()

array([2018, 2019], dtype=int32)

In [63]:
# Check the months represented
df_transaction["DATE"].dt.month_name().unique()

array(['October', 'May', 'August', 'June', 'September', 'February',
       'March', 'November', 'April', 'July', 'January', 'December'],
      dtype=object)

#Phase 3: Data Quality Assessment

In [64]:
df_transaction[['LYLTY_CARD_NBR','TXN_ID']].head()

,LYLTY_CARD_NBR,TXN_ID
0,1000,1
1,1307,348
2,1343,383
3,2373,974
4,2426,1038


In [65]:
# Check if the columns are unique
df_transaction["TXN_ID"].is_unique

False

In [66]:
df_transaction["LYLTY_CARD_NBR"].is_unique

False

In [67]:
df_transaction["TXN_ID"].duplicated().sum()

np.int64(1708)

In [68]:
df_transaction["LYLTY_CARD_NBR"].duplicated().sum()

np.int64(192198)

In [69]:
duplicated_txn = df_transaction[df_transaction["TXN_ID"].duplicated(keep=False)]
duplicated_txn.head()

,DATE,STORE_NBR,LYLTY_CARD_NBR,TXN_ID,PROD_NBR,PROD_NAME,PROD_QTY,TOT_SALES
41,2019-05-20,55,55073,48887,4,Dorito Corn Chp Supreme 380g,1,3.25
42,2019-05-20,55,55073,48887,113,Twisties Chicken270g,1,4.60
376,2019-01-10,7,7364,7739,50,Tostitos Lightly Salted 175g,2,8.80
377,2019-01-10,7,7364,7739,20,Doritos Cheese Supreme 330g,2,11.40
418,2018-10-18,12,12301,10982,50,Tostitos Lightly Salted 175g,2,8.80


In [70]:
duplicated_txn["TXN_ID"].unique()[:10]

array([48887,  7739, 10982, 14546, 16683, 42616, 44177, 48663, 48884,
       53351])

In [71]:
df_transaction[df_transaction["TXN_ID"] == 48887]

,DATE,STORE_NBR,LYLTY_CARD_NBR,TXN_ID,PROD_NBR,PROD_NAME,PROD_QTY,TOT_SALES
41,2019-05-20,55,55073,48887,4,Dorito Corn Chp Supreme 380g,1,3.25
42,2019-05-20,55,55073,48887,113,Twisties Chicken270g,1,4.60


#Phase 4: Feature Engineering

In [72]:
df_transaction["Year"] = df_transaction["DATE"].dt.year

df_transaction["Month"] = df_transaction["DATE"].dt.month

df_transaction["Month_Name"] = df_transaction["DATE"].dt.month_name()

df_transaction["Quarter"] = df_transaction["DATE"].dt.quarter

In [73]:
# List of ID columns
id_columns = [
    "LYLTY_CARD_NBR",
    "TXN_ID",
    "STORE_NBR",
    "PROD_NBR",
    "TOT_SALES"
]

# Check for negative and zero values
for col in id_columns:
    print(f"\nColumn: {col}")
    print(f"Negative values: {(df_transaction[col] < 0).sum()}")
    print(f"Zero values: {(df_transaction[col] == 0).sum()}")
    print("-" * 40)


Column: LYLTY_CARD_NBR
Negative values: 0
Zero values: 0
----------------------------------------

Column: TXN_ID
Negative values: 0
Zero values: 0
----------------------------------------

Column: STORE_NBR
Negative values: 0
Zero values: 0
----------------------------------------

Column: PROD_NBR
Negative values: 0
Zero values: 0
----------------------------------------

Column: TOT_SALES
Negative values: 0
Zero values: 0
----------------------------------------


In [74]:
df_transaction["PROD_NAME"].unique()

array(['Natural Chip        Compny SeaSalt175g',
       'CCs Nacho Cheese    175g',
       'Smiths Crinkle Cut  Chips Chicken 170g',
       'Smiths Chip Thinly  S/Cream&Onion 175g',
       'Kettle Tortilla ChpsHny&Jlpno Chili 150g',
       'Old El Paso Salsa   Dip Tomato Mild 300g',
       'Smiths Crinkle Chips Salt & Vinegar 330g',
       'Grain Waves         Sweet Chilli 210g',
       'Doritos Corn Chip Mexican Jalapeno 150g',
       'Grain Waves Sour    Cream&Chives 210G',
       'Kettle Sensations   Siracha Lime 150g',
       'Twisties Cheese     270g', 'WW Crinkle Cut      Chicken 175g',
       'Thins Chips Light&  Tangy 175g', 'CCs Original 175g',
       'Burger Rings 220g', 'NCC Sour Cream &    Garden Chives 175g',
       'Doritos Corn Chip Southern Chicken 150g',
       'Cheezels Cheese Box 125g', 'Smiths Crinkle      Original 330g',
       'Infzns Crn Crnchers Tangy Gcamole 110g',
       'Kettle Sea Salt     And Vinegar 175g',
       'Smiths Chip Thinly  Cut Original 175g', 'K

In [75]:
df_purchase["LIFESTAGE"].unique()

array(['YOUNG SINGLES/COUPLES', 'YOUNG FAMILIES', 'OLDER SINGLES/COUPLES',
       'MIDAGE SINGLES/COUPLES', 'NEW FAMILIES', 'OLDER FAMILIES',
       'RETIREES'], dtype=object)

In [76]:
df_purchase["PREMIUM_CUSTOMER"].unique()

array(['Premium', 'Mainstream', 'Budget'], dtype=object)

#Feature Engineering

In [77]:
df_transaction["PROD_NAME"].tail(20)

,PROD_NAME
264816,Cobs Popd Sea Salt Chips 110g
264817,Sunbites Whlegrn Crisps Frch/Onin 90g
264818,Infuzions SourCream&Herbs Veg Strws 110g
264819,Kettle Original 175g
264820,Pringles Mystery Flavour 134g
264821,Kettle Sea Salt And Vinegar 175g
264822,Pringles Slt Vingar 134g
264823,Kettle 135g Swt Pot Sea Salt
264824,Kettle Tortilla ChpsBtroot&Ricotta 150g
264825,Infuzions BBQ Rib Prawn Crackers 110g


In [78]:
df_transaction["PACKET_SIZE_G"] = df_transaction["PROD_NAME"].str.extract(r"(\d+)").astype(int)

In [79]:
df_transaction["PACKET_SIZE_G"].head(10)

,PACKET_SIZE_G
0,175
1,175
2,170
3,175
4,150
5,300
6,330
7,210
8,150
9,210


In [80]:
brand_lookup = {
    "Smiths": "Smiths",
    "Smith": "Smiths",

    "Kettle": "Kettle",

    "Doritos": "Doritos",
    "Dorito": "Doritos",

    "Pringles": "Pringles",

    "Twisties": "Twisties",

    "CCs": "CCs",

    "Grain Waves": "Grain Waves",
    "GrnWves": "Grain Waves",

    "Natural Chip Co": "Natural Chip Co",
    "Natural ChipCo": "Natural Chip Co",
    "Natural Chip": "Natural Chip Co",

    "Old El Paso": "Old El Paso",

    "Red Rock Deli": "Red Rock Deli",
    "RRD": "Red Rock Deli",

    "Thins": "Thins",

    "Cheezels": "Cheezels",

    "Infuzions": "Infuzions",
    "Infzns": "Infuzions",

    "Cobs": "Cobs",

    "Woolworths": "Woolworths",
    "WW": "Woolworths",

    "Burger Rings": "Burger Rings",

    "French Fries": "French Fries",

    "Tostitos": "Tostitos",

    "Tyrrells": "Tyrrells",

    "Cheetos": "Cheetos",

    "Sunbites": "Sunbites",
    "Snbts": "Sunbites",

    "NCC": "NCC"
}

In [81]:
df_transaction[df_transaction["PROD_NAME"].str.contains("NCC")]

,DATE,STORE_NBR,LYLTY_CARD_NBR,TXN_ID,PROD_NBR,PROD_NAME,PROD_QTY,TOT_SALES,Year,Month,Month_Name,Quarter,PACKET_SIZE_G
17,2018-08-14,22,22411,18646,98,NCC Sour Cream & Garden Chives 175g,1,3.0,2018,8,August,3,175
21,2018-08-16,33,33081,29949,98,NCC Sour Cream & Garden Chives 175g,1,3.0,2018,8,August,3,175
437,2018-08-28,13,13176,12211,98,NCC Sour Cream & Garden Chives 175g,2,6.0,2018,8,August,3,175
535,2019-01-16,22,22131,18288,98,NCC Sour Cream & Garden Chives 175g,2,6.0,2019,1,January,1,175
828,2019-02-18,41,41087,38000,98,NCC Sour Cream & Garden Chives 175g,2,6.0,2019,2,February,1,175
...,...,...,...,...,...,...,...,...,...,...,...,...,...
264307,2018-07-03,248,248244,250500,98,NCC Sour Cream & Garden Chives 175g,2,6.0,2018,7,July,3,175
264360,2019-03-05,249,249354,251273,98,NCC Sour Cream & Garden Chives 175g,1,3.0,2019,3,March,1,175
264636,2019-06-11,264,264268,263027,98,NCC Sour Cream & Garden Chives 175g,1,3.0,2019,6,June,2,175
264721,2018-08-21,266,266329,264152,98,NCC Sour Cream & Garden Chives 175g,1,3.0,2018,8,August,3,175


In [82]:
def extract_brand(product_name):
    for brand, standard_name in brand_lookup.items():
        if product_name.startswith(brand):
            return standard_name
    return "Unknown"

In [83]:
df_transaction["BRAND_NAME"] = df_transaction["PROD_NAME"].apply(extract_brand)

In [84]:
df_transaction["BRAND_NAME"].value_counts()

,count
BRAND_NAME,
Kettle,41288
Smiths,31822
Doritos,28147
Pringles,25102
Red Rock Deli,17779
Woolworths,14757
Infuzions,14201
Thins,14075
Cobs,9693


#Phase 4: Exploratory Data Analysis

##Section 1: Overall Business Overview

In [85]:
df_transaction.head()

,DATE,STORE_NBR,LYLTY_CARD_NBR,TXN_ID,PROD_NBR,PROD_NAME,PROD_QTY,TOT_SALES,Year,Month,Month_Name,Quarter,PACKET_SIZE_G,BRAND_NAME
0,2018-10-17,1,1000,1,5,Natural Chip Compny SeaSalt175g,2,6.0,2018,10,October,4,175,Natural Chip Co
1,2019-05-14,1,1307,348,66,CCs Nacho Cheese 175g,3,6.3,2019,5,May,2,175,CCs
2,2019-05-20,1,1343,383,61,Smiths Crinkle Cut Chips Chicken 170g,2,2.9,2019,5,May,2,170,Smiths
3,2018-08-17,2,2373,974,69,Smiths Chip Thinly S/Cream&Onion 175g,5,15.0,2018,8,August,3,175,Smiths
4,2018-08-18,2,2426,1038,108,Kettle Tortilla ChpsHny&Jlpno Chili 150g,3,13.8,2018,8,August,3,150,Kettle


In [86]:
revenue = df_transaction["TOT_SALES"].sum()
print(f"Total Revenue: $ {revenue:.0f}")

Total Revenue: $ 1934409


In [87]:
transactions = df_transaction["TXN_ID"].sum()
print(f"Total Transactions: {transactions:.0f}")

Total Transactions: 35794677941


In [88]:
df_transaction.head()

,DATE,STORE_NBR,LYLTY_CARD_NBR,TXN_ID,PROD_NBR,PROD_NAME,PROD_QTY,TOT_SALES,Year,Month,Month_Name,Quarter,PACKET_SIZE_G,BRAND_NAME
0,2018-10-17,1,1000,1,5,Natural Chip Compny SeaSalt175g,2,6.0,2018,10,October,4,175,Natural Chip Co
1,2019-05-14,1,1307,348,66,CCs Nacho Cheese 175g,3,6.3,2019,5,May,2,175,CCs
2,2019-05-20,1,1343,383,61,Smiths Crinkle Cut Chips Chicken 170g,2,2.9,2019,5,May,2,170,Smiths
3,2018-08-17,2,2373,974,69,Smiths Chip Thinly S/Cream&Onion 175g,5,15.0,2018,8,August,3,175,Smiths
4,2018-08-18,2,2426,1038,108,Kettle Tortilla ChpsHny&Jlpno Chili 150g,3,13.8,2018,8,August,3,150,Kettle


In [89]:
total_transactions = df_transaction["TXN_ID"].nunique()

print(f"Total Transactions: {total_transactions:,}")

Total Transactions: 263,127


In [90]:
total_units = df_transaction["PROD_QTY"].sum()
print(f"Total Units Sold: {total_units:,}")

Total Units Sold: 505,122


In [91]:
avg_spend = revenue/total_transactions
print(f"Average Spend per Transaction: ${avg_spend:.2f}")

Average Spend per Transaction: $7.35


In [92]:
unique_customers = df_transaction["LYLTY_CARD_NBR"].nunique()

print(f"Unique Customers: {unique_customers:,}")

Unique Customers: 72,637


In [93]:
purchase_frequency = total_transactions/unique_customers
print(f"Purchase Frequecy: {purchase_frequency:1f}")

Purchase Frequecy: 3.622493


#Section 1: Product Analysis

In [94]:
#Which brands generate the most revenue?
df_transaction.groupby("BRAND_NAME")["TOT_SALES"].sum().sort_values(ascending=False)

,TOT_SALES
BRAND_NAME,
Kettle,390239.8
Doritos,241890.9
Smiths,224654.2
Pringles,177655.5
Infuzions,99047.6
Red Rock Deli,95046.0
Old El Paso,90785.1
Thins,88852.5
Twisties,81522.1


In [95]:
#Which packet sizes are most popular?

df_transaction.groupby("PACKET_SIZE_G")["PROD_QTY"].sum().sort_values(ascending=False)

,PROD_QTY
PACKET_SIZE_G,
175,126465
150,82174
134,48019
110,42835
170,38088
165,29051
300,28813
330,23999
380,12673


In [96]:
#Which packet sizes generate the most revenue?

df_transaction.groupby("PACKET_SIZE_G")["TOT_SALES"].sum().sort_values(ascending=False)

,TOT_SALES
PACKET_SIZE_G,
175,485431.4
150,304288.5
134,177655.5
110,162765.4
170,146673.0
330,136794.3
300,113330.6
165,101360.6
380,76719.6


In [97]:
#Which products are purchased most frequently?

df_transaction.groupby("PROD_NAME")["TXN_ID"].count().sort_values(ascending=False)

,TXN_ID
PROD_NAME,
Kettle Mozzarella Basil & Pesto 175g,3304
Kettle Tortilla ChpsHny&Jlpno Chili 150g,3296
Cobs Popd Swt/Chlli &Sr/Cream Chips 110g,3269
Tyrrells Crisps Ched & Chives 165g,3268
Cobs Popd Sea Salt Chips 110g,3265
...,...
RRD Pc Sea Salt 165g,1431
Woolworths Medium Salsa 300g,1430
NCC Sour Cream & Garden Chives 175g,1419


#Section 2: Customer Analysis

In [98]:
df_transaction = df_transaction.merge(df_purchase, on="LYLTY_CARD_NBR", how="left")

In [99]:
df_transaction[["LYLTY_CARD_NBR", "LIFESTAGE", "PREMIUM_CUSTOMER"]].head()
df_transaction[["LIFESTAGE", "PREMIUM_CUSTOMER"]].isna().sum()

,0
LIFESTAGE,0
PREMIUM_CUSTOMER,0


In [100]:
# Which customer segment buys the most chips?
customer_buys_more = df_transaction.groupby("LIFESTAGE")["PROD_QTY"].sum().sort_values(ascending=False)
customer_buys_more

,PROD_QTY
LIFESTAGE,
OLDER SINGLES/COUPLES,104201
OLDER FAMILIES,94992
RETIREES,94166
YOUNG FAMILIES,84561
YOUNG SINGLES/COUPLES,66634
MIDAGE SINGLES/COUPLES,47721
NEW FAMILIES,12847


In [101]:
# Which customer segment generates the most revenue?
customer_spends_more = df_transaction.groupby("LIFESTAGE")["TOT_SALES"].sum().sort_values(ascending=False)
customer_spends_more

,TOT_SALES
LIFESTAGE,
OLDER SINGLES/COUPLES,402420.75
RETIREES,366470.90
OLDER FAMILIES,353767.20
YOUNG FAMILIES,316160.10
YOUNG SINGLES/COUPLES,260405.30
MIDAGE SINGLES/COUPLES,184751.30
NEW FAMILIES,50433.45


In [102]:
# Which customer segment purchases most frequently?
customer_freq = df_transaction.groupby("LIFESTAGE")["TXN_ID"].count().sort_values(ascending=False)
customer_freq

,TXN_ID
LIFESTAGE,
OLDER SINGLES/COUPLES,54478
RETIREES,49763
OLDER FAMILIES,48596
YOUNG FAMILIES,43592
YOUNG SINGLES/COUPLES,36377
MIDAGE SINGLES/COUPLES,25110
NEW FAMILIES,6919


In [103]:
# Which segment spends the most per transaction?
customer_spend_per_transaction = df_transaction.groupby("LIFESTAGE")["TOT_SALES"].sum() / df_transaction.groupby("LIFESTAGE")["TXN_ID"].count()
customer_spend_per_transaction.sort_values(ascending=False)

,0
LIFESTAGE,
OLDER SINGLES/COUPLES,7.386849
RETIREES,7.364325
MIDAGE SINGLES/COUPLES,7.357678
NEW FAMILIES,7.289124
OLDER FAMILIES,7.279760
YOUNG FAMILIES,7.252709
YOUNG SINGLES/COUPLES,7.158515


#Section 4: Purchasing Behaviour Analysis

In [104]:
# Which brands are preferred by each customer segment?
df_transaction.groupby(["LIFESTAGE", "BRAND_NAME"])["PROD_QTY"].sum().sort_values(ascending=False)

LIFESTAGE              BRAND_NAME  
OLDER SINGLES/COUPLES  Kettle          17024
RETIREES               Kettle          15568
OLDER FAMILIES         Kettle          13367
                       Smiths          12502
YOUNG FAMILIES         Kettle          12153
                                       ...  
NEW FAMILIES           Cheetos           106
                       Sunbites          104
                       Burger Rings       70
                       French Fries       56
                       NCC                50
Name: PROD_QTY, Length: 154, dtype: int64

In [105]:
# Which packet sizes are preferred by each customer segment?
df_transaction.groupby(["LIFESTAGE", "PACKET_SIZE_G"])["PROD_QTY"].sum().sort_values(ascending=False)

LIFESTAGE              PACKET_SIZE_G
OLDER SINGLES/COUPLES  175              26042
OLDER FAMILIES         175              23806
RETIREES               175              23556
YOUNG FAMILIES         175              21466
OLDER SINGLES/COUPLES  150              17053
                                        ...  
NEW FAMILIES           90                 104
                       125                 98
                       220                 70
                       180                 54
                       70                  53
Name: PROD_QTY, Length: 147, dtype: int64

In [106]:
# Do Premium customers spend more?
df_transaction.groupby("PREMIUM_CUSTOMER")["TOT_SALES"].sum().sort_values(ascending=False)

,TOT_SALES
PREMIUM_CUSTOMER,
Mainstream,750744.50
Budget,676211.55
Premium,507452.95


# Section 5: Sales Trend Analysis

In [107]:
df_transaction.columns

Index(['DATE', 'STORE_NBR', 'LYLTY_CARD_NBR', 'TXN_ID', 'PROD_NBR',
       'PROD_NAME', 'PROD_QTY', 'TOT_SALES', 'Year', 'Month', 'Month_Name',
       'Quarter', 'PACKET_SIZE_G', 'BRAND_NAME', 'LIFESTAGE',
       'PREMIUM_CUSTOMER'],
      dtype='object')

In [108]:
# Are there seasonal trends?
df_transaction.groupby("Month_Name")["TOT_SALES"].sum().sort_values(ascending=False)

,TOT_SALES
Month_Name,
December,167913.40
March,166265.20
July,165275.30
October,164409.70
January,162642.30
June,160538.60
September,160522.00
November,160233.70
April,159845.10


In [109]:
# Which quarter generated the most revenue?
df_transaction.groupby("Quarter")["TOT_SALES"].sum().sort_values(ascending=False)

,TOT_SALES
Quarter,
4,492556.80
3,484528.35
1,479572.50
2,477751.35


In [110]:
# Did sales improve over time?
monthly_units = (
    df_transaction.groupby("Month")["PROD_QTY"]
    .sum()
    .sort_index()
)

monthly_units

,PROD_QTY
Month,
1,42501
2,39220
3,43347
4,41825
5,41300
6,41852
7,43242
8,41484
9,41792


In [111]:
df_transaction["PACKET_SIZE_G"].unique()

array([175, 170, 150, 300, 330, 210, 270, 220, 125, 110, 134, 380, 180,
       165, 135, 250, 200, 160, 190,  90,  70])

In [113]:
df_transaction.to_csv("chip_sales_transaction.csv",index=False)